# Job Interview Guide Workshop — ML Specialist

**Interview result:** 15/15 (100%). No incorrect answers were identified.

**Purpose:** Since no weak topics were found in the multiple-choice assessment, this notebook provides practical mastery checks instead of inventing weaknesses. Inspired by the uploaded workshop notebooks: concept → code → experiment → interpretation → reflection.

**Learning objectives**
1. Identify features and targets and avoid data leakage.
2. Implement and evaluate regression with MSE and R².
3. Evaluate classification with log-loss, recall and precision.
4. Compare KNN and decision trees with reproducible pipelines.
5. Explain model selection trade-offs to a technical interviewer.

**Instructions:** Run cells in order. Complete each `TODO` before reading its sanity check. No external data or network access required.

## 0. Environment and reproducibility
The source workshops use Python, NumPy, pandas, scikit-learn, and notebook-based exploration. Set a random seed and use bundled scikit-learn datasets to keep this exercise reproducible.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, r2_score, log_loss, recall_score, precision_score, confusion_matrix
RANDOM_STATE = 42
print("NumPy:", np.__version__)
print("Ready for workshop")

NumPy: 2.5.3
Ready for workshop


## 1. Features, target, and leakage
**Concept:** Features (`X`) are the predictors; the target (`y`) is the quantity to predict. Fit learned preprocessing transformations on training data only. In scikit-learn, a `Pipeline` ensures that scaling is learned within each training fold during cross-validation.

**Exercise:** Load diabetes data and create an 80/20 split. Do not use the target as a feature.

In [2]:
diabetes = load_diabetes(as_frame=True, scaled=False)
df = diabetes.frame.copy()
print(df.head())
print("Rows, columns:", df.shape)
# TODO 1: Replace the following placeholders.
X = None  # e.g. df.drop(columns=[...])
y = None  # e.g. df[...]
# TODO 2: Create X_train, X_test, y_train, y_test with test_size=0.2 and random_state=RANDOM_STATE.
# X_train, X_test, y_train, y_test = ...

    age  sex   bmi     bp     s1     s2    s3   s4      s5    s6  target
0  59.0  2.0  32.1  101.0  157.0   93.2  38.0  4.0  4.8598  87.0   151.0
1  48.0  1.0  21.6   87.0  183.0  103.2  70.0  3.0  3.8918  69.0    75.0
2  72.0  2.0  30.5   93.0  156.0   93.6  41.0  4.0  4.6728  85.0   141.0
3  24.0  1.0  25.3   84.0  198.0  131.4  40.0  5.0  4.8903  89.0   206.0
4  50.0  1.0  23.0  101.0  192.0  125.4  52.0  4.0  4.2905  80.0   135.0
Rows, columns: (442, 11)


In [3]:
# Sanity checks: run after completing the TODOs.
if X is not None and y is not None and 'X_train' in globals():
    assert 'target' not in X.columns
    assert len(X) == len(y) == 442
    assert len(X_train) + len(X_test) == 442
    assert set(X_train.index).isdisjoint(set(X_test.index))
    print("PASS: no target leakage and disjoint splits")
else:
    print("Complete TODOs in section 1 first.")

Complete TODOs in section 1 first.


## 2. Linear regression: predictions, MSE and R²
**Concept:** Linear regression predicts continuous values. MSE averages squared residuals, while R² compares predictive squared error with a mean-only baseline. A higher test R² is generally preferable when comparing models on the same evaluation data.

**Exercise:** Fit a model and calculate both metrics manually or using scikit-learn.

In [4]:
if 'X_train' in globals():
    reg = LinearRegression()
    # TODO 3: Fit reg on the training data and create y_pred on X_test.
    # reg.fit(...)
    # y_pred = ...
    # TODO 4: Compute MSE and R².
    # mse = ...
    # r2 = ...
    print("Training rows:", len(X_train))
else:
    print("Complete section 1 first.")

Complete section 1 first.


In [5]:
# Sanity check for Q5: prices are in thousands of dollars.
actual = np.array([200., 300., 400.])
predicted = np.array([210., 290., 420.])
assert np.isclose(mean_squared_error(actual, predicted), 200.0)
print("Reference MSE:", mean_squared_error(actual, predicted))
# TODO 5: Write a NumPy expression for MSE without calling sklearn.

Reference MSE: 200.0


## 3. Logistic regression: probability, threshold, and log-loss
**Concept:** Logistic regression models a binary outcome. Sigmoid maps a real-valued score to a probability. Binary cross-entropy penalizes confident incorrect probabilities. Choosing a threshold affects precision and recall.

**Exercise:** Calculate sigmoid and per-example log-loss for a positive observation.

In [6]:
def sigmoid(z):
    z = np.asarray(z, dtype=float)
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))
assert np.isclose(sigmoid(0), 0.5)
print("sigmoid(0):", sigmoid(0))
# TODO 6: Calculate log-loss for y=1 at p=0.90 and p=0.20.
# Use -np.log(p) or sklearn.metrics.log_loss with labels=[0,1].
# loss_a = ...
# loss_b = ...
# assert loss_a < loss_b

sigmoid(0): 0.5


## 4. KNN: scaling and hyperparameter tuning
**Concept:** Euclidean-distance KNN is sensitive to numerical scales. Tune `n_neighbors` on training data using cross-validation, with scaling inside the pipeline. Never choose K based on the test set.

**Exercise:** Build a small synthetic binary dataset, then compare k values.

In [7]:
X_cls, y_cls = make_classification(n_samples=400, n_features=6, n_informative=4,
                                    n_redundant=0, weights=[0.8, 0.2], random_state=RANDOM_STATE)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls, y_cls, test_size=0.25, stratify=y_cls, random_state=RANDOM_STATE)
knn_pipe = Pipeline([('scaler', StandardScaler()), ('classifier', KNeighborsClassifier())])
# TODO 7: Define GridSearchCV with parameter grid {'classifier__n_neighbors':[1,3,5,9,15]},
# cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE), scoring='f1'.
# Fit on Xc_train/yc_train, and print best_params_ and test recall.
assert set(np.unique(yc_train)) == {0, 1}
print("Train/test sizes:", len(yc_train), len(yc_test))

Train/test sizes: 300 100


## 5. Decision trees: leaf predictions
**Concept:** A squared-error regression tree predicts the average target value in a leaf. A classification tree predicts the most frequent class by default, and class probabilities reflect proportions of training examples in the leaf.

**Exercise:** Reproduce the interview leaf examples and inspect the results.

In [8]:
leaf_prices = np.array([200, 240, 260, 300])
leaf_labels = np.array([0]*15 + [1]*5)
# TODO 8: Compute the regression leaf prediction with np.mean.
# leaf_prediction = ...
# TODO 9: Compute probability of class 1 and the majority class.
# p_default = ...
# predicted_class = ...
assert np.isclose(np.mean(leaf_prices), 250)
assert np.isclose(np.mean(leaf_labels), 0.25)
print("Expected regression leaf mean:", np.mean(leaf_prices))

Expected regression leaf mean: 250.0


## 6. Scenario: imbalanced fraud detection
**Concept:** Accuracy can be misleading for rare events. Recall measures the fraction of actual positive cases found. Precision measures the fraction of predicted positives that are truly positive. Select metrics based on business costs.

**Exercise:** Compare two classifiers on a realistic imbalanced synthetic dataset. Use a held-out test set only for final reporting.

In [9]:
fraud_X, fraud_y = make_classification(n_samples=1000, n_features=8, n_informative=5,
    n_redundant=0, weights=[0.95, 0.05], flip_y=0, random_state=RANDOM_STATE)
Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    fraud_X, fraud_y, test_size=0.25, stratify=fraud_y, random_state=RANDOM_STATE)
models = {
    'logistic': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),
    'tree': DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
}
# TODO 10: Fit both models, predict on Xf_test, then print confusion_matrix,
# precision_score(zero_division=0) and recall_score for each.
print("Test fraud prevalence:", yf_test.mean())

Test fraud prevalence: 0.048


## 7. Interview reflection (write in your own words)
1. Why must a scaler be fitted on the training data rather than the complete dataset?
2. Explain MSE and R² to a nontechnical manager in two sentences.
3. When could higher recall be worth lower precision? Give a business example.
4. Explain why a very deep decision tree might perform worse on unseen data.
5. What is one concept you understand better after completing the code exercises?

**Exit ticket:** Choose one model from this notebook, explain why it fits its prediction target, identify a possible failure mode, and name a metric you would monitor in production.

**Next step:** Commit the completed notebook and add a brief README with environment requirements, execution instructions, and results.